# TextCNN (사전학습 한국어 fastText 임베딩)

- 딥러닝 비교군 (교수 피드백 반영, 학습 임베딩 계열)
- 입력: 결합 텍스트, 토큰화: 공백 분리 (보고서에 명시)
- 임베딩: fastText `cc.ko.300.vec` 로 초기화, OOV는 랜덤
- kernel [3,4,5] x 100 filters, Dropout 0.3, class weight
- **사전 준비:** `artifacts/cc.ko.300.vec` 파일 필요 (Phase 2 셋업에서 다운로드)

In [ ]:
import os, time, numpy as np, pandas as pd

# 스모크 테스트: True면 클래스별 소량 샘플만 사용 (파이프라인 점검용)
SMOKE_TEST = False
N_SMOKE = 1000

def find_root(start='.'):
    p = os.path.abspath(start)
    while p != os.path.dirname(p):
        if os.path.exists(os.path.join(p, 'processed_data')):
            return p
        p = os.path.dirname(p)
    return os.path.abspath(start)

ROOT = find_root()
DATA_DIR = os.path.join(ROOT, 'processed_data')
ART_DIR = os.path.join(ROOT, 'artifacts')
os.makedirs(ART_DIR, exist_ok=True)
print('ROOT     :', ROOT)
print('DATA_DIR :', DATA_DIR)
print('ART_DIR  :', ART_DIR)

In [ ]:
# 라벨 순서 (label_mapping.csv 기준: 인덱스 = label 값)
LABEL_NAMES = ['가사', '개인정보/ICT', '근로자', '금융조세', '기업',
               '민사', '특허/저작권', '행정', '형사A(생활형)', '형사B(일반형)']
N_CLASS = 10

In [ ]:
import re, torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from sklearn.metrics import f1_score, accuracy_score, classification_report
device = 'cuda' if torch.cuda.is_available() else 'cpu'
MAX_LEN = 400
MIN_FREQ = 2
EMB_DIM = 300
FREEZE_EMB = False
FASTTEXT_PATH = os.path.join(ART_DIR, 'cc.ko.300.vec')
print('device:', device)

In [ ]:
def load_split(name):
    """결합 입력(text = 판시사항 + 요약문)과 라벨 반환."""
    df = pd.read_csv(os.path.join(DATA_DIR, name + '.csv'))
    if SMOKE_TEST:
        per = max(2, N_SMOKE // N_CLASS)
        df = df.groupby('label', group_keys=False).apply(
            lambda d: d.sample(min(len(d), per), random_state=42))
    text = (df['jdgmn'].fillna('') + ' ' + df['summ_pass'].fillna('')).tolist()
    labels = df['label'].to_numpy()
    return text, labels

In [ ]:
def tok(s):
    s = str(s).lower()
    s = re.sub(r'[^0-9a-z가-힣\s]', ' ', s)
    return s.split()

tr_text, ytr = load_split('train')
val_text, yval = load_split('val')
te_text, yte = load_split('test')
tr_tok = [tok(t) for t in tr_text]
val_tok = [tok(t) for t in val_text]
te_tok = [tok(t) for t in te_text]
print('train docs:', len(tr_tok))

In [ ]:
cnt = Counter(w for doc in tr_tok for w in doc)
itos = ['<pad>', '<unk>'] + [w for w, c in cnt.items() if c >= MIN_FREQ]
stoi = {w: i for i, w in enumerate(itos)}
print('vocab size:', len(itos))

In [ ]:
# fastText 임베딩 매트릭스 구성 (vocab 단어만)
from gensim.models import KeyedVectors
kv = KeyedVectors.load_word2vec_format(FASTTEXT_PATH, limit=300000)
rng = np.random.default_rng(42)
emb_matrix = rng.normal(0, 0.1, (len(itos), EMB_DIM)).astype('float32')
emb_matrix[0] = 0.0
hit = 0
for w, i in stoi.items():
    if w in kv:
        emb_matrix[i] = kv[w]; hit += 1
print(f'fastText 매칭: {hit}/{len(itos)} ({hit/len(itos)*100:.1f}%)')

In [ ]:
def encode(doc):
    ids = [stoi.get(w, 1) for w in doc[:MAX_LEN]]
    ids = ids + [0] * (MAX_LEN - len(ids))
    return ids

class DS(Dataset):
    def __init__(self, toks, y):
        self.X = [encode(d) for d in toks]; self.y = y
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return torch.tensor(self.X[i]), torch.tensor(int(self.y[i]))

tr_loader = DataLoader(DS(tr_tok, ytr), batch_size=128, shuffle=True)

In [ ]:
class TextCNN(nn.Module):
    def __init__(self, emb, n_cls=N_CLASS, ks=(3, 4, 5), nf=100, p=0.3, freeze=False):
        super().__init__()
        V, D = emb.shape
        self.emb = nn.Embedding(V, D, padding_idx=0)
        self.emb.weight.data.copy_(torch.tensor(emb))
        self.emb.weight.requires_grad = not freeze
        self.convs = nn.ModuleList([nn.Conv1d(D, nf, k, padding=k // 2) for k in ks])
        self.drop = nn.Dropout(p)
        self.fc = nn.Linear(nf * len(ks), n_cls)
    def forward(self, x):
        e = self.emb(x).transpose(1, 2)
        outs = [F.relu(c(e)).max(dim=2)[0] for c in self.convs]
        return self.fc(self.drop(torch.cat(outs, 1)))

In [ ]:
counts = np.bincount(ytr, minlength=N_CLASS)
w = counts.sum() / (N_CLASS * np.maximum(counts, 1))
class_weight = torch.tensor(w, dtype=torch.float32).to(device)

torch.manual_seed(42)
net = TextCNN(emb_matrix, freeze=FREEZE_EMB).to(device)
crit = nn.CrossEntropyLoss(weight=class_weight)
opt = torch.optim.Adam(filter(lambda p: p.requires_grad, net.parameters()), lr=1e-3)

def evaluate(toks, y):
    net.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(toks), 256):
            xb = torch.tensor([encode(d) for d in toks[i:i+256]]).to(device)
            preds.append(net(xb).argmax(1).cpu().numpy())
    return np.concatenate(preds)

In [ ]:
best_f1, best_state, patience, bad = -1, None, 6, 0
for epoch in range(40):
    net.train()
    for xb, yb in tr_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad(); loss = crit(net(xb), yb); loss.backward(); opt.step()
    vp = evaluate(val_tok, yval)
    f1 = f1_score(yval, vp, average='macro')
    print(f'epoch {epoch:02d}  val F1-macro {f1:.4f}')
    if f1 > best_f1:
        best_f1 = f1; bad = 0
        best_state = {k: v.cpu().clone() for k, v in net.state_dict().items()}
    else:
        bad += 1
        if bad >= patience:
            print('early stop'); break
net.load_state_dict(best_state)
print('best val F1-macro:', round(best_f1, 4))

In [ ]:
yp = evaluate(te_tok, yte)
print(f'TEST  Accuracy {accuracy_score(yte, yp):.4f}  |  F1-macro {f1_score(yte, yp, average="macro"):.4f}  (baseline 0.6347)')
print(classification_report(yte, yp, target_names=LABEL_NAMES, digits=4))
torch.save(net.state_dict(), os.path.join(ART_DIR, 'textcnn.pt'))